# Test payload sizes for predict_sensa_aml_scores

Loads feature list from `model_info`, builds payloads with the correct feature count. The server accepts at most **441 records per request** (nginx body limit). We test **1k, 10k, 50k, 100k, 500k, and 1M** total records by batching requests and measure total time and throughput.

## 1. Config and load model_info (features)

In [ ]:
import json
import os
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# --- Config ---
BASE_URL = os.getenv("BASE_URL", "")
DEPLOYMENT = "typology-circular"
ENDPOINT = f"{BASE_URL}/models/{DEPLOYMENT}/predict_sensa_aml_scores"
DEPLOYMENT_API_KEY = os.getenv("DEPLOYMENT_API_KEY", "")

# Max records per request (nginx body limit). Batches use this size.
MAX_BATCH_SIZE = 441
# Number of parallel workers (should match number of replicas for optimal utilization)
MAX_WORKERS = 1

# Model info path: data/model_info.json (single model) or model_infos/model_info.json (model_infos array)
MODEL_INFO_PATHS = [
    os.path.join("data", "model_info.json"),
    os.path.join("model_infos", "model_info.json"),
]

def load_features_from_model_info():
    for path in MODEL_INFO_PATHS:
        if not os.path.isfile(path):
            continue
        with open(path, "r") as f:
            data = json.load(f)
        # Single model format: {"features": {"all_features": [...]}}
        if "features" in data and "all_features" in data["features"]:
            return data["features"]["all_features"]
        # model_infos array: {"model_infos": [{"features": {"all_features": [...]}, ...}]}
        if "model_infos" in data and data["model_infos"]:
            first = data["model_infos"][0]
            if "features" in first and "all_features" in first["features"]:
                return first["features"]["all_features"]
    raise FileNotFoundError(
        f"No model_info found. Tried: {MODEL_INFO_PATHS}"
    )

feature_names = load_features_from_model_info()
num_features = len(feature_names)
print(f"Loaded {num_features} features from model_info")
print(f"First 5: {feature_names[:5]}")

## 2. Build payload (N records, correct feature count per row)

In [ ]:
def build_payload(n_records: int, num_features: int, seed: int = 42):
    """Build payload with n_records rows, each row has num_features values (placeholder floats)."""
    import random
    rng = random.Random(seed)
    # Use small floats to keep payload size reasonable; model may expect normalized values
    features = [
        [round(rng.uniform(0, 1), 6) for _ in range(num_features)]
        for _ in range(n_records)
    ]
    return {"features": features}

# Example: one full batch (max allowed per request)
payload = build_payload(MAX_BATCH_SIZE, num_features)
payload_size_mb = len(json.dumps(payload)) / (1024 * 1024)
print(f"Max batch: {MAX_BATCH_SIZE} records x {num_features} features")
print(f"Approx. JSON size: {payload_size_mb:.2f} MB")
print(f"First row (first 6 values): {payload['features'][0][:6]}")

## 3. Single run — one batch (max 441 records)

One request with `MAX_BATCH_SIZE` records to verify the endpoint and measure per-batch time.

**If you get 500 Internal Server Error:** The upstream model service failed (not nginx). Common causes: (1) **Feature count/order mismatch** — the deployed model expects a different number or order of features than your `model_info` (e.g. test2-deployment trained with different schema). (2) **Pod crash/OOM** — check the deployment pod logs in Kubernetes. (3) **Wrong API shape** — try the "Debug 500" cell below (minimal payload, response headers).

In [ ]:
payload = build_payload(MAX_BATCH_SIZE, num_features)
payload_size_mb = len(json.dumps(payload)) / (1024 * 1024)
print(f"Payload: {MAX_BATCH_SIZE} records, ~{payload_size_mb:.2f} MB")

start = time.perf_counter()
resp = requests.post(
    ENDPOINT,
    json=payload,
    headers={"X-API-Key": DEPLOYMENT_API_KEY,"Content-Type": "application/json"},
    timeout=300,
)
elapsed = time.perf_counter() - start

print(f"Status: {resp.status_code}")
print(f"Time: {elapsed:.2f} s")
if resp.ok:
    data = resp.json()
    # Show keys and maybe length of predictions if present
    if isinstance(data, dict):
        for k, v in list(data.items())[:8]:
            if isinstance(v, (list, dict)) and len(v) <= 3:
                print(f"  {k}: {v}")
            elif hasattr(v, "__len__"):
                print(f"  {k}: <len={len(v)}>")
            else:
                print(f"  {k}: {v}")
else:
    print(f"Body: {resp.text[:500]}")
    if resp.headers:
        print("Response headers:", dict(resp.headers))

### Debug 500: minimal request and /predict

**500 on both test-deployment and test2-deployment** usually means a **shared** issue: (1) **feature count mismatch** — deployed models expect a different number of features than your local `model_info` (240), or (2) **gateway/upstream** — check pod logs for the real traceback.

Run the cell below: it sends **1 record** to `predict_sensa_aml_scores` and to **/predict**. If both return 500 → feature count or service error (check pod logs). If /predict returns 200 but predict_sensa_aml_scores returns 500 → issue is specific to the sensa endpoint (e.g. model or COPOD).

In [ ]:
# # Minimal payload: 1 row x num_features
# minimal = build_payload(1, num_features)
# base = f"{BASE_URL}/models/{DEPLOYMENT}"

# # 1) predict_sensa_aml_scores with 1 record
# r_sensa = requests.post(f"{base}/predict_sensa_aml_scores", json=minimal, headers={"Content-Type": "application/json"}, timeout=60)
# print(f"predict_sensa_aml_scores (1 row): {r_sensa.status_code}")
# if r_sensa.ok:
#     print("  keys:", list(r_sensa.json().keys()))
# else:
#     print("  body:", (r_sensa.text[:200] if r_sensa.text else ""))

# # 2) /predict with 1 record — if this works, the service is up and accepts our feature count
# r_predict = requests.post(f"{base}/predict", json=minimal, headers={"Content-Type": "application/json"}, timeout=60)
# print(f"predict (1 row): {r_predict.status_code}")
# if r_predict.ok:
#     print("  keys:", list(r_predict.json().keys()))
# else:
#     print("  body:", (r_predict.text[:200] if r_predict.text else ""))

# # If both 500 → check deployment pod logs (feature count or model error). If only sensa 500 → sensa/COPOD issue.

## 4. Batched runs: 1k, 10k, 50k, … 1M records (parallel)

Total records are split into batches of `MAX_BATCH_SIZE` (441). Batches are processed in parallel using `MAX_WORKERS` threads (configured to match replica count). We measure total time and throughput for each target size.

In [ ]:
def create_session_with_retries():
    """Create a requests session with retry strategy and connection pooling."""
    session = requests.Session()
    # Retry strategy: retry on connection errors, timeouts, and 5xx errors
    retry_strategy = Retry(
        total=3,  # Total retries
        backoff_factor=1,  # Wait 1s, 2s, 4s between retries
        status_forcelist=[500, 502, 503, 504],  # Retry on these HTTP status codes
        allowed_methods=["POST"],  # Only retry POST requests
    )
    adapter = HTTPAdapter(
        max_retries=retry_strategy,
        pool_connections=MAX_WORKERS * 2,  # Connection pool size
        pool_maxsize=MAX_WORKERS * 2,
    )
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session

# Create a shared session for connection pooling (thread-safe)
_shared_session = None

def get_session():
    """Get or create the shared session (lazy initialization)."""
    global _shared_session
    if _shared_session is None:
        _shared_session = create_session_with_retries()
    return _shared_session

def send_batch_request(batch_idx: int, batch_size: int, total_records: int, num_features: int):
    """Send a single batch request with retry logic. Returns (batch_idx, success, error_msg)."""
    session = get_session()
    pl = build_payload(batch_size, num_features, seed=42 + batch_idx)
    
    # Retry logic for connection errors
    max_retries = 3
    for attempt in range(max_retries):
        try:
            r = session.post(ENDPOINT, json=pl, headers={"X-API-Key": DEPLOYMENT_API_KEY,"Content-Type": "application/json"}, timeout=300)
            if not r.ok:
                # Don't retry on client errors (4xx), only on server errors
                if 400 <= r.status_code < 500:
                    return (batch_idx, False, f"HTTP {r.status_code}")
                # Retry on server errors
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)  # Exponential backoff
                    continue
                return (batch_idx, False, f"HTTP {r.status_code}")
            return (batch_idx, True, None)
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as e:
            # Retry on connection errors
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)  # Exponential backoff: 1s, 2s, 4s
                continue
            return (batch_idx, False, f"Connection error: {str(e)}")
        except Exception as e:
            return (batch_idx, False, str(e))
    return (batch_idx, False, "Max retries exceeded")

# Total record counts to test (batched with MAX_BATCH_SIZE = 441 per request)
TOTAL_RECORDS_TO_TEST = [1_000, 10_000, 50_000, 100_000, 500_000, 1_000_000]
results = []
print(f"{DEPLOYMENT} (parallel with {MAX_WORKERS} workers)")

## 4b. Model time responses for MAX_WORKERS = [1, 3, 5, 10] (save to file)

Runs the same batched benchmark for each worker count, collects all results, and saves to `model_time_response_results.json` and `model_time_response_results.csv`.

In [ ]:
# WORKER_COUNTS = [3, 5, 10, 15, 20]
WORKER_COUNTS = [16, 17]

RESULTS_JSON = f"{DEPLOYMENT.replace('-', '_')}_model_time_response_results.json"
RESULTS_CSV = f"{DEPLOYMENT.replace('-', '_')}_model_time_response_results.csv"

all_results = []

for workers in WORKER_COUNTS:
    # Use this worker count for session pool and thread pool; reset session so new pool size is used
    MAX_WORKERS = workers
    _shared_session = None

    print(f"{DEPLOYMENT} (parallel with {MAX_WORKERS} workers)")
    for total in TOTAL_RECORDS_TO_TEST:
        n_batches = (total + MAX_BATCH_SIZE - 1) // MAX_BATCH_SIZE
        start = time.perf_counter()

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = []
            for i in range(n_batches):
                size = min(MAX_BATCH_SIZE, total - i * MAX_BATCH_SIZE)
                future = executor.submit(send_batch_request, i, size, total, num_features)
                futures.append(future)

            failed_batches = []
            for future in as_completed(futures):
                batch_idx, success, error_msg = future.result()
                if not success:
                    failed_batches.append((batch_idx + 1, error_msg))

            if failed_batches:
                print(f"  {total:>8} records: {len(failed_batches)} batch(es) failed:")
                for batch_num, error_msg in failed_batches[:5]:
                    print(f"    Batch {batch_num}/{n_batches}: {error_msg}")
                if len(failed_batches) > 5:
                    print(f"    ... and {len(failed_batches) - 5} more failures")
                failed = True
            else:
                failed = False

        elapsed = time.perf_counter() - start
        error_messages = [f"Batch {b}/{n_batches}: {msg}" for b, msg in failed_batches] if failed_batches else []

        row = {
            "total_records": total,
            "n_batches": n_batches,
            "batch_size": MAX_BATCH_SIZE,
            "workers": MAX_WORKERS,
            "time_s": round(elapsed, 2),
            "success": not failed,
            "failed_batches_count": len(failed_batches) if failed_batches else 0,
            "error_messages": error_messages,
        }
        if not failed:
            row["records_per_sec"] = round(total / elapsed, 1)
            row["ms_per_record"] = round(elapsed * 1000 / total, 2)
            print(f"  {total:>8} records -> {n_batches:>5} batches -> {elapsed:>8.2f} s | {total/elapsed:.1f} rec/s | {elapsed*1000/total:.2f} ms/rec")
        else:
            row["records_per_sec"] = None
            row["ms_per_record"] = None
        all_results.append(row)
    print()

# Save results
with open(RESULTS_JSON, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"Saved {len(all_results)} rows to {RESULTS_JSON}")

import pandas as pd
df = pd.DataFrame(all_results)
df.to_csv(RESULTS_CSV, index=False)
print(f"Saved to {RESULTS_CSV}")
df

## 5. (Optional) Single batched run — collect predictions (parallel)

Run a chosen total (e.g. 10k or 1M) in batches of 441 with parallel processing. Optionally collect prediction lists from the response. Adapt the response key (`probabilities`, `copod_scores`, etc.) if needed.

In [ ]:
def send_batch_and_collect(batch_idx: int, batch_size: int, num_features: int):
    """Send a batch request with retry logic and return predictions. Returns (batch_idx, predictions, error)."""
    session = get_session()
    pl = build_payload(batch_size, num_features, seed=42 + batch_idx)
    
    # Retry logic for connection errors
    max_retries = 3
    for attempt in range(max_retries):
        try:
            r = session.post(ENDPOINT, json=pl, headers={"X-API-Key": DEPLOYMENT_API_KEY,"Content-Type": "application/json"}, timeout=300)
            if not r.ok:
                # Don't retry on client errors (4xx), only on server errors
                if 400 <= r.status_code < 500:
                    return (batch_idx, None, f"HTTP {r.status_code}")
                # Retry on server errors
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                return (batch_idx, None, f"HTTP {r.status_code}")
            data = r.json()
            preds = data.get("probabilities", data.get("predictions", data.get("scores", [])))
            return (batch_idx, preds if isinstance(preds, list) else None, None)
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as e:
            # Retry on connection errors
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
                continue
            return (batch_idx, None, f"Connection error: {str(e)}")
        except Exception as e:
            return (batch_idx, None, str(e))
    return (batch_idx, None, "Max retries exceeded")

TOTAL_RECORDS = 10_000  # or 1_000_000 for a long run
n_batches = (TOTAL_RECORDS + MAX_BATCH_SIZE - 1) // MAX_BATCH_SIZE

start = time.perf_counter()
all_probabilities = []
batch_results = {}  # Store results by batch_idx to maintain order

# Submit all batches to thread pool
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = []
    for i in range(n_batches):
        size = min(MAX_BATCH_SIZE, TOTAL_RECORDS - i * MAX_BATCH_SIZE)
        future = executor.submit(send_batch_and_collect, i, size, num_features)
        futures.append((i, future))
    
    # Collect results as they complete
    for batch_idx, future in futures:
        batch_idx_result, preds, error = future.result()
        if error:
            print(f"Batch {batch_idx_result+1}/{n_batches} failed: {error}")
            break
        if preds is not None:
            batch_results[batch_idx_result] = preds

# Combine predictions in order
for i in range(len(batch_results)):
    if i in batch_results:
        all_probabilities.extend(batch_results[i])

elapsed = time.perf_counter() - start

print(f"Batches: {n_batches} x up to {MAX_BATCH_SIZE} (parallel, {MAX_WORKERS} workers) -> {len(all_probabilities)} predictions")
print(f"Total time: {elapsed:.2f} s ({elapsed * 1000 / len(all_probabilities):.2f} ms/record)" if all_probabilities else "No predictions collected.")

In [ ]:
# Optional: pandas table
import pandas as pd
pd.DataFrame(results)